### Import

In [ ]:
import argparse, os, sys, datetime, glob, importlib
import random
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.trainer import Trainer
from omegaconf import OmegaConf

from taming.main import instantiate_from_config
from taming.modules.diffusionmodules.model import Encoder, Decoder
from taming.modules.vqvae.quantize import VectorQuantizer2 as VectorQuantizer
from taming.modules.vqvae.quantize import GumbelQuantize, EMAVectorQuantizer

#### Configuration

In [ ]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

base = "../taming/configs/brats_f16_vqgan.yaml"
config = OmegaConf.load(base)

model = instantiate_from_config(config.model)

## Stage 1

In [ ]:
# if you don't have the tokens, you should tokenize the image first
# For tokenizing, using generate_eval.py

data_path = "../outputs/llabit__eval.pickle"

with open(data_path, 'rb') as f:
    data = pickle.load(f)

In [ ]:
def get_image(tokens):
    tokens = torch.tensor(tokens).cuda()
    z = model.quantize.get_codebook_entry(tokens, shape=(1,12,12,-1))

    img = model.decode(z)
    img = img.squeeze().permute(1,2,0).detach().cpu().numpy()
    img = np.clip(img, -1., 1.)
    img = (img + 1.) / 2.

    return img

In [ ]:
for sub in data:
    tokens = sub["gen_image"]

    t2_img = get_image(tokens)
    sub["gen_image_t2"] = t2_img

    break

### Save results

In [ ]:
model = model.cuda()
def save_pickle(data_path, save_path):
    with open(data_path, 'rb') as f:
        data = pickle.load(f)

    for sub in data:
        tokens = sub["gen_image"]

        t2_img = get_image(tokens)
        sub["gen_image_t2"] = t2_img

    with open(save_path, 'wb') as f:
        pickle.dump(data, f)

## Stage 2

#### Configuration

In [ ]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

base = "../taming/configs/brats_f16_vqgan2.yaml"
config = OmegaConf.load(base)

model = instantiate_from_config(config.model)

In [ ]:
data_path = "../outputs/llabit__eval_165498_i2i_0.5.pickle"

with open(data_path, 'rb') as f:
    data = pickle.load(f)

### Image Translation

In [ ]:
from PIL import Image

def preprocess_image(image_path):
    image = Image.open(image_path)
    if not image.mode == "RGB":
        image = image.convert("RGB")
    image = np.array(image).astype(np.uint8)
    image = (image/127.5 - 1.0).astype(np.float32)
    return image

def get_image(src_path, tokens, target):
    src_path = src_path.replace("store8", "store4")
    src = preprocess_image(src_path)
    src = torch.tensor(src).permute(2,0,1).unsqueeze(0).cuda()
    hs = model.encode(src)
    cond = model.cond_stage_model(target).cuda()

    tokens = torch.tensor(tokens).cuda()

    img = model(src, tokens, cond)
    img = img.squeeze().permute(1,2,0).detach().cpu().numpy()
    img = np.clip(img, -1., 1.)
    img = (img + 1.0) / 2.0

    return img

In [ ]:
model = model.cuda()
for sub in data:
    src_path = sub["path"]
    tokens = sub["gen_image"]

    t2_img = get_image(src_path, tokens, ['t2'])
    sub["gen_image_t2"] = t2_img

    break

#### Save results

In [ ]:
model = model.cuda()
def save_pickle(data_path, save_path):
    with open(data_path, 'rb') as f:
        data = pickle.load(f)

    for sub in data:
        src_path = sub["path"]
        tokens = sub["gen_image"]

        t2_img = get_image(src_path, tokens, "T2")
        sub["gen_image_t2"] = t2_img

    with open(save_path, 'wb') as f:
        pickle.dump(data, f)